# 04 · Modeling & Evaluation (Deliverable D + prediction file)
**Addis Ride Demand Forecasting · Qiyas Hackathon**

Run order: `01_cleaning_and_integration` → `02_analysis_report` → `03_visualizations` → **`04_modeling_and_evaluation`** (this notebook).

**Inputs** (all in `data/processed/`, produced by Deliverable A): `master_train.csv`, `master_test.csv`, `train_ops.csv` (used *only* for the leakage demo in D4).
**Outputs:** `submission/team_<NAME>_submission.csv`, `models/final_model.joblib`, `figures/fig10…fig12`, `reports/D_model_evaluation.md`, `reports/D_tables/*.csv`.

> Set `FAST=1` in the environment for a ~2-minute dry run (few trees, 2 tuning trials) to check the notebook end-to-end before the real run.

---
## Model selection (read this first)

| Candidate | Strengths for *this* task | Weaknesses | Verdict |
|---|---|---|---|
| **Mean / seasonal-naive** | Required baselines (D1); seasonal-naive already captures hour × weekday × zone rhythm | Blind to weather, events, trend | Baselines only |
| **Ridge / linear** | Fast, interpretable, can partly extrapolate the trend | Cannot learn zone × hour × event interactions without heavy hand-crafting | Compare (D2) |
| **Random forest** | Handles interactions, robust | Slower, cannot extrapolate beyond training range, big file | Compare (D2) |
| **HistGradientBoosting (sklearn)** | Strong tabular model, no extra dependency | Fewer tuning options and no native sample-weighted categorical handling as flexible as LightGBM | Compare (D2) |
| **LightGBM (gradient boosting)** | Best accuracy/speed on tabular data; native categorical `zone`; `poisson` objective suits non-negative counts; handles mixed 0/1 event flags + continuous lags; fast enough for time-ordered tuning; small `joblib` file for the demo | Trees can't extrapolate a trend by themselves → solved by the `level_*` / `lag_*` features already in your master table | **Chosen final model** |
| **MLP (neural net)** | Allowed, can model smooth effects | Needs scaling, noisy on 83k rows, slower to tune, usually loses to boosting here | Compare (D2) |

**Why LightGBM is the pick.** (1) The data is tabular with ~60 mixed features and strong interactions (event window × zone × hour; rain × zone type) — the sweet spot for boosted trees. (2) The target is a non-negative count, so a `poisson` objective is a natural option that we let the time-ordered search decide on. (3) Trend: demand grows ~36 % Jan→Oct and trees cannot extrapolate, but `level_28d`, `level_7d` and the 14/21/28-day lags give the model the *current level*; the optional "ratio target" experiment in D8 tests predicting trips relative to that level. (4) The rules require a final model that is validated chronologically, tuned with time-ordered folds, and saved with `joblib` — all easy here.

**Design decisions baked into the notebook**
- `month` is **excluded**: November is never seen in training, so a tree would treat it like October, silently acting as a wrong trend proxy.
- `zone` is a native categorical for LightGBM (one-hot for sklearn models).
- Tuning folds and evaluation folds use **disjoint 14-day validation windows**, so the reported rolling-origin scores are not tuned on.
- The master table's lag/level features are all ≥ 14 days old, i.e. they are available for **every** hour of a 14-day forecast horizon.

## 0 · Setup

In [ ]:
import os, time, json, warnings, platform, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import ParameterSampler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

try:
    from IPython.display import display
except Exception:
    display = print
try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
except ImportError as e:
    raise ImportError("LightGBM is required:  pip install lightgbm") from e

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
FAST = os.environ.get("FAST", "0") == "1"          # quick dry-run switch
TEAM_NAME = "your_team_name"                        # <-- TODO: lowercase, underscores, no spaces

# ---- relative paths only (works from notebooks/ or from the project root)
ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent
PROC, MODELS, FIGS = ROOT / "data/processed", ROOT / "models", ROOT / "figures"
REPORTS, SUB = ROOT / "reports", ROOT / "submission"
TABLES = REPORTS / "D_tables"
for d in (MODELS, FIGS, REPORTS, SUB, TABLES):
    d.mkdir(parents=True, exist_ok=True)

# ---- consistent, colour-blind-safe style (Okabe-Ito)
OKABE = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9", "#F0E442", "#000000"]
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 200, "font.size": 11, "axes.titlesize": 13,
                     "axes.labelsize": 11, "legend.fontsize": 10, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.prop_cycle": matplotlib.cycler(color=OKABE)})

print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__,
      "| lightgbm", lgb.__version__, "| FAST =", FAST)
print("project root:", ROOT)

In [ ]:
train = pd.read_csv(PROC / "master_train.csv", parse_dates=["ts"]).sort_values(["ts", "zone"]).reset_index(drop=True)
test  = pd.read_csv(PROC / "master_test.csv",  parse_dates=["ts"]).sort_values(["ts", "zone"]).reset_index(drop=True)
ZONES = sorted(train["zone"].unique())
print("train", train.shape, train.ts.min(), "->", train.ts.max())
print("test ", test.shape,  test.ts.min(),  "->", test.ts.max())
print("zones:", ZONES)

# ---------- feature groups (names come from data_dictionary_master.csv)
TARGET   = "trips"
CAL      = ["hour", "dow", "is_weekend", "hour_of_week", "day_of_month",
            "is_payday_window", "is_holiday", "is_holiday_eve", "is_school_break"]
TREND    = ["lag_n_missing", "lag_336h", "lag_504h", "lag_672h",
            "same_how_mean", "level_28d", "level_7d", "level_ratio_7_28"]
WEATHER  = ["temp_c", "rain_mm", "humidity_pct", "wind_kmh", "rain_3h", "rain_6h", "rain_24h",
            "rain_class", "rain_missing"]
EVENTS   = [c for c in train.columns if c.startswith("ev_")]
FULL     = CAL + TREND + WEATHER + EVENTS               # 'zone' is added separately
EXCLUDED = {"row_id": "identifier", "ts": "raw timestamp (its parts are already in CAL)",
            "trips": "target",
            "month": "test month (11) never appears in train -> trees would treat it as October (fake trend proxy)"}

# the same feature columns must exist in train and test
assert [c for c in test.columns if c != "row_id"] == [c for c in train.columns if c not in ("row_id", TARGET)], "train/test columns differ"
assert set(FULL) | set(EXCLUDED) | {"zone"} == set(train.columns), set(train.columns) ^ (set(FULL) | set(EXCLUDED) | {"zone"})
assert train[FULL].isna().sum().sum() == 0 and test[FULL].isna().sum().sum() == 0, "NaN in features"
print(f"\n{len(FULL)} model features + zone | calendar {len(CAL)}, trend/lag {len(TREND)}, weather {len(WEATHER)}, events {len(EVENTS)}")

In [ ]:
# ---------- helpers
def rmse(y, p): return float(np.sqrt(np.mean((np.asarray(y, float) - np.asarray(p, float)) ** 2)))
def mae(y, p):  return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(p, float))))

HORIZON    = pd.Timedelta(days=14)
MAIN_CUT   = pd.Timestamp("2025-10-18")                       # train < 18 Oct, validate 18-31 Oct
TUNE_CUTS  = [pd.Timestamp(c) for c in ["2025-07-05", "2025-08-02", "2025-08-30", "2025-09-27"]]   # D6 only
EVAL_CUTS  = [pd.Timestamp(c) for c in ["2025-06-21", "2025-07-19", "2025-08-16", "2025-09-13", "2025-10-18"]]  # D3/D5/D7
# validation windows of TUNE_CUTS and EVAL_CUTS do not overlap -> reported rolling scores are not tuned on
for a in TUNE_CUTS:
    for b in EVAL_CUTS:
        assert abs((a - b).days) >= 14, (a, b)

def split(cut):
    cut = pd.Timestamp(cut)
    tr = train[train.ts < cut]
    va = train[(train.ts >= cut) & (train.ts < cut + HORIZON)]
    return tr, va

REPORT = []                                                  # collected for reports/D_model_evaluation.md
def add_report(title, df=None, text=""):
    REPORT.append((title, df, text))
    if df is not None:
        fn = "".join(ch if ch.isalnum() else "_" for ch in title.lower()).strip("_")[:60]
        df.to_csv(TABLES / f"{fn}.csv", index=False)

def md_table(df, nd=3):
    cols = list(df.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(f"{v:.{nd}f}" if isinstance(v, (float, np.floating)) else str(v) for v in r) + " |")
    return "\n".join(out)

tr0, va0 = split(MAIN_CUT)
print(f"MAIN SPLIT  train: {tr0.ts.min()} -> {tr0.ts.max()} ({len(tr0):,} rows)")
print(f"            valid: {va0.ts.min()} -> {va0.ts.max()} ({len(va0):,} rows)")

### Model code (LightGBM wrapper + sklearn pipelines)
`fit_gbm` / `predict_gbm` support two target modes: `raw` (predict trips) and `ratio` (predict trips ÷ `level_28d`, weighted by the level, then multiply back — tested in D8).

In [ ]:
def prep(df, feats):
    X = df[["zone"] + list(feats)].copy()
    X["zone"] = pd.Categorical(X["zone"], categories=ZONES)          # same category list for train & test
    return X

DEFAULT_PARAMS = dict(objective="regression", n_estimators=800, learning_rate=0.03, num_leaves=63,
                      min_child_samples=40, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                      reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbose=-1)

def _p(params):
    p = dict(params)
    if FAST:
        p["n_estimators"] = min(p.get("n_estimators", 100), 60)
    return p

def fit_gbm(df_tr, feats, params=None, mode="raw"):
    m = LGBMRegressor(**_p(params or DEFAULT_PARAMS))
    X = prep(df_tr, feats)
    if mode == "raw":
        m.fit(X, df_tr[TARGET])
    else:
        lvl = df_tr["level_28d"].clip(lower=1.0)
        m.fit(X, df_tr[TARGET] / lvl, sample_weight=lvl)
    return m

def predict_gbm(m, df, feats, mode="raw"):
    p = m.predict(prep(df, feats))
    if mode == "ratio":
        p = p * df["level_28d"].clip(lower=1.0).values
    return np.clip(p, 0, None)                                      # trips can't be negative

def sk_pipe(est, feats=None):
    feats = feats or FULL
    pre = ColumnTransformer([("zone", OneHotEncoder(handle_unknown="ignore"), ["zone"]),
                             ("num", StandardScaler(), list(feats))])
    return Pipeline([("pre", pre), ("est", est)])

def Xsk(df, feats=None):
    X = df[["zone"] + list(feats or FULL)].copy()
    X["zone"] = X["zone"].astype(str)
    return X

def run_folds(feats, params=None, mode="raw", cuts=EVAL_CUTS, keep_oof=False):
    # train on everything before each cut, validate on the next 14 days
    rows, oofs = [], []
    for c in cuts:
        tr, va = split(c)
        m = fit_gbm(tr, feats, params, mode)
        p = predict_gbm(m, va, feats, mode)
        rows.append(dict(cut=str(pd.Timestamp(c).date()), n_val=len(va), mean_trips=va[TARGET].mean(),
                         rmse=rmse(va[TARGET], p), mae=mae(va[TARGET], p)))
        if keep_oof:
            oofs.append(va.assign(pred=p, fold=str(pd.Timestamp(c).date())))
    return pd.DataFrame(rows), (pd.concat(oofs, ignore_index=True) if keep_oof else None)

## D1 · Baselines
Two required baselines on the chronological validation fortnight (train < 18 Oct, validate 18–31 Oct):
1. **Mean predictor** – the mean of `trips` over the training period.
2. **Seasonal-naive** – mean trips for the same zone × hour-of-week; we report both the *whole training period* and the *most recent 8 weeks* (the latter follows the trend better).

In [ ]:
def seasonal_naive(tr, va, weeks=None):
    ref = tr if weeks is None else tr[tr.ts >= tr.ts.max() - pd.Timedelta(weeks=weeks)]
    tab = ref.groupby(["zone", "hour_of_week"])[TARGET].mean().rename("p").reset_index()
    p = va[["zone", "hour_of_week"]].merge(tab, on=["zone", "hour_of_week"], how="left")["p"]
    p = p.fillna(va["zone"].map(ref.groupby("zone")[TARGET].mean())).fillna(tr[TARGET].mean())
    return p.values

base = []
p_mean = np.full(len(va0), tr0[TARGET].mean())
p_sn_all = seasonal_naive(tr0, va0, None)
p_sn_8w  = seasonal_naive(tr0, va0, 8)
for name, p in [("Mean predictor", p_mean), ("Seasonal-naive (all train)", p_sn_all), ("Seasonal-naive (last 8 weeks)", p_sn_8w)]:
    base.append(dict(baseline=name, rmse=rmse(va0[TARGET], p), mae=mae(va0[TARGET], p)))
D1 = pd.DataFrame(base)
display(D1)
BASE_MEAN_RMSE, BASE_SN_RMSE = D1.rmse[0], D1.rmse[2]            # the one marked in fig10 (recent 8 weeks)
add_report("D1 Baselines", D1, f"Validation {va0.ts.min().date()} to {va0.ts.max().date()}. The seasonal-naive baseline removes "
           f"{100*(1-D1.rmse[2]/D1.rmse[0]):.0f}% of the mean-predictor RMSE, so most of the signal is the weekly rhythm per zone; "
           "everything a model adds must beat this.")

**✍️ Interpretation (add 1–2 sentences after running):** …

## D4 · Feature availability & leakage audit
Every candidate column and whether it is known **at forecast time** (1–14 Nov). Then a deliberate leakage test: we add `active_drivers` and `avg_wait_min` (same-hour, from `train_ops.csv`) and show the inflated validation score.

In [ ]:
rows = []
def grp(cols, group, source, known, why, decision):
    for c in cols:
        rows.append(dict(feature=c, group=group, source=source, known_at_forecast_time=known, reason=why, decision=decision))
grp(CAL[:5], "calendar", "ts", "yes", "pure function of the timestamp", "used")
grp(["is_payday_window"], "calendar", "ts", "yes", "function of day of month", "used")
grp(["is_holiday", "is_holiday_eve", "is_school_break"], "calendar", "events_calendar (scheduled)", "yes", "official holidays / school terms are published in advance", "used")
grp(WEATHER, "weather", "weather_hourly", "yes (forecast)", "Nov rows are *forecasts* (noisier than observed history) - see weather stress test", "used")
grp(EVENTS, "events", "events_calendar", "yes (scheduled)", "confirmed events are announced ahead; cancelled ones excluded in A", "used")
grp(["lag_336h", "lag_504h", "lag_672h", "same_how_mean", "level_28d", "level_7d", "level_ratio_7_28", "lag_n_missing"],
    "trend/lag", "trip history", "yes", "every lag/level is >= 14 days old, so it exists for every hour of a 14-day horizon", "used")
grp(["zone"], "zone", "all tables", "yes", "fixed label", "used")
rows += [dict(feature="month", group="calendar", source="ts", known_at_forecast_time="yes", reason="value 11 never seen in train; acts as a wrong trend proxy", decision="EXCLUDED"),
         dict(feature="avg_wait_min", group="operations", source="train_ops", known_at_forecast_time="NO", reason="a consequence of demand (high demand -> longer waits)", decision="EXCLUDED"),
         dict(feature="active_drivers", group="operations", source="train_ops", known_at_forecast_time="NO", reason="supply responds to demand; unknown for next Tuesday 18:00", decision="EXCLUDED"),
         dict(feature="avg_fare_birr", group="operations", source="train_ops", known_at_forecast_time="NO", reason="only exists after trips happen", decision="EXCLUDED (used only to price revenue in the demo)"),
         dict(feature="row_id / ts / trips", group="id/target", source="-", known_at_forecast_time="n/a", reason="identifier, raw timestamp, target", decision="EXCLUDED")]
D4 = pd.DataFrame(rows)
print(D4.groupby(["group", "known_at_forecast_time", "decision"]).size().rename("n_features"))
add_report("D4 Feature availability audit", D4, "Only forecast-time-safe columns enter the model. Operational columns are excluded because they are effects of demand, not causes.")

In [ ]:
# ---- deliberate leaky experiment (NEVER used in the final model)
ops = pd.read_csv(PROC / "train_ops.csv", usecols=["row_id", "avg_wait_min", "active_drivers"])
leak = train.merge(ops, on="row_id", how="left")
for c in ["avg_wait_min", "active_drivers"]:
    leak[c] = leak[c].fillna(leak[c].median())
LEAK = ["avg_wait_min", "active_drivers"]
ltr, lva = leak[leak.ts < MAIN_CUT], leak[(leak.ts >= MAIN_CUT) & (leak.ts < MAIN_CUT + HORIZON)]
t0 = time.time()
m_leak = LGBMRegressor(**_p(DEFAULT_PARAMS)).fit(prep(ltr, FULL + LEAK), ltr[TARGET])
p_leak = np.clip(m_leak.predict(prep(lva, FULL + LEAK)), 0, None)
m_clean = fit_gbm(tr0, FULL); p_clean = predict_gbm(m_clean, va0, FULL)
D4b = pd.DataFrame([dict(model="LightGBM, clean features", rmse=rmse(va0[TARGET], p_clean), mae=mae(va0[TARGET], p_clean)),
                    dict(model="LightGBM + active_drivers + avg_wait_min (LEAKY)", rmse=rmse(lva[TARGET], p_leak), mae=mae(lva[TARGET], p_leak))])
display(D4b)
add_report("D4b Leakage experiment", D4b,
           f"Adding same-hour driver/wait columns drops RMSE from {D4b.rmse[0]:.2f} to {D4b.rmse[1]:.2f} - a fake gain: in production "
           "next week's drivers and waits are unknown, so this model could not be run.")

**✍️ Interpretation:** the leaky score looks better only because drivers/wait are a *result* of demand. …

## D2 · Model comparison
Same split, same features (`FULL` + zone), RMSE / MAE / training time. Four model families beyond the baselines: ridge, random forest, sklearn HistGradientBoosting, an MLP neural net — plus LightGBM with default settings (tuned later in D6).

In [ ]:
cands = {
    "Ridge (linear)":   sk_pipe(Ridge(alpha=1.0, random_state=SEED)),
    "Random forest":    sk_pipe(RandomForestRegressor(n_estimators=40 if FAST else 300, min_samples_leaf=3, max_features=0.5, n_jobs=-1, random_state=SEED)),
    "HistGradientBoosting": sk_pipe(HistGradientBoostingRegressor(max_iter=60 if FAST else 500, learning_rate=0.05, max_leaf_nodes=31, random_state=SEED)),
    "MLP (neural net)": sk_pipe(MLPRegressor(hidden_layer_sizes=(128, 64), early_stopping=True, n_iter_no_change=8,
                                             max_iter=15 if FAST else 150, random_state=SEED)),
}
rows = []
for name, pipe in cands.items():
    t0 = time.time(); pipe.fit(Xsk(tr0), tr0[TARGET]); secs = time.time() - t0
    p = np.clip(pipe.predict(Xsk(va0)), 0, None)
    rows.append(dict(model=name, rmse=rmse(va0[TARGET], p), mae=mae(va0[TARGET], p), train_seconds=secs))
    print(f"{name:24s} RMSE {rows[-1]['rmse']:.3f}  MAE {rows[-1]['mae']:.3f}  ({secs:.1f}s)")
t0 = time.time(); m_def = fit_gbm(tr0, FULL, DEFAULT_PARAMS); secs = time.time() - t0
p = predict_gbm(m_def, va0, FULL)
rows.append(dict(model="LightGBM (default)", rmse=rmse(va0[TARGET], p), mae=mae(va0[TARGET], p), train_seconds=secs))
D2 = pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)
D2["rmse_vs_seasonal_naive_%"] = 100 * (D2.rmse / BASE_SN_RMSE - 1)
display(D2)
WINNER = D2.model[0]
print("\nWinner on the validation fortnight:", WINNER)
add_report("D2 Model comparison", D2, f"Winner: {WINNER}. All models use identical features and the same chronological split ({va0.ts.min().date()} to {va0.ts.max().date()}).")

**✍️ Interpretation / why the winner won (add after running):** …  *(If the winner is not LightGBM, that is fine — choose by evidence and update `FINAL_FAMILY` accordingly; the tuning and saving code below assumes LightGBM.)*

## D5 · Ablation — was each join worth it?
Same model type (LightGBM, default parameters) and identical rolling folds. (i) calendar + zone + trend only → (ii) + weather → (iii) + events → (iv) + both.
Note: `is_holiday`, `is_holiday_eve`, `is_school_break` count as *calendar*; "events" means the `ev_*` window features.

In [ ]:
variants = {"(i) calendar + zone + trend": CAL + TREND,
            "(ii) + weather":             CAL + TREND + WEATHER,
            "(iii) + events":             CAL + TREND + EVENTS,
            "(iv) + weather + events":    CAL + TREND + WEATHER + EVENTS}
abl, per_fold = [], {}
for name, feats in variants.items():
    res, _ = run_folds(feats, DEFAULT_PARAMS, "raw", EVAL_CUTS)
    per_fold[name] = res
    abl.append(dict(variant=name, n_features=len(feats) + 1, rmse_mean=res.rmse.mean(), rmse_sd=res.rmse.std(ddof=1),
                    mae_mean=res.mae.mean(), rmse_holdout_18oct=res.rmse.iloc[-1]))
D5 = pd.DataFrame(abl)
D5["d_rmse_vs_(i)"] = D5.rmse_mean - D5.rmse_mean.iloc[0]
D5["d_rmse_%"] = 100 * D5["d_rmse_vs_(i)"] / D5.rmse_mean.iloc[0]
D5["d_mae_vs_(i)"] = D5.mae_mean - D5.mae_mean.iloc[0]
display(D5)
add_report("D5 Ablation", D5,
           f"Weather alone changes mean RMSE by {D5['d_rmse_%'][1]:+.1f}%, events alone by {D5['d_rmse_%'][2]:+.1f}%, both by {D5['d_rmse_%'][3]:+.1f}% "
           "(negative = better). Means over the 5 rolling folds.")

**✍️ Interpretation:** which join paid off most, which was marginal, and is that consistent with B2/B3? …

## D6 · Hyper-parameter tuning (time-ordered)
Random search, objective = mean RMSE across 4 **chronological** folds (`TUNE_CUTS`, each train-before-cut → next 14 days). The 18 Oct holdout and the D3 windows are never used for tuning. The search also decides between `regression` (L2) and `poisson` objectives.

In [ ]:
SPACE = {"objective": ["regression", "poisson"],
         "num_leaves": [15, 31, 63, 127],
         "learning_rate": [0.02, 0.03, 0.05, 0.08],
         "n_estimators": [300, 600, 900, 1200],
         "min_child_samples": [10, 20, 40, 80],
         "subsample": [0.6, 0.8, 1.0],
         "colsample_bytree": [0.5, 0.7, 0.9],
         "reg_lambda": [0.0, 1.0, 5.0, 20.0]}
FIXED = dict(subsample_freq=1, random_state=SEED, n_jobs=-1, verbose=-1)
N_TRIALS = 2 if FAST else 30
trials = list(ParameterSampler(SPACE, n_iter=N_TRIALS, random_state=SEED))
print(f"search space size {np.prod([len(v) for v in SPACE.values()]):,} | trials {N_TRIALS} | tuning folds {[str(c.date()) for c in TUNE_CUTS]}")

log = []
t0 = time.time()
res, _ = run_folds(FULL, DEFAULT_PARAMS, "raw", TUNE_CUTS)
log.append(dict(trial="default", **{k: DEFAULT_PARAMS[k] for k in SPACE}, tune_rmse=res.rmse.mean()))
for i, t in enumerate(trials, 1):
    res, _ = run_folds(FULL, {**FIXED, **t}, "raw", TUNE_CUTS)
    log.append(dict(trial=i, **t, tune_rmse=res.rmse.mean()))
    print(f"trial {i:2d}/{N_TRIALS}  tune-RMSE {log[-1]['tune_rmse']:.3f}  {t}")
D6log = pd.DataFrame(log).sort_values("tune_rmse").reset_index(drop=True)
print(f"search took {(time.time()-t0)/60:.1f} min")
best_row = D6log.iloc[0]
BEST_PARAMS = {**FIXED, **{k: (best_row[k].item() if hasattr(best_row[k], "item") else best_row[k]) for k in SPACE}}
print("\nBEST PARAMS:", BEST_PARAMS)

In [ ]:
# score before vs after on the untouched 18-Oct holdout
m_before = fit_gbm(tr0, FULL, DEFAULT_PARAMS); p_b = predict_gbm(m_before, va0, FULL)
m_after  = fit_gbm(tr0, FULL, BEST_PARAMS);    p_a = predict_gbm(m_after,  va0, FULL)
D6 = pd.DataFrame([dict(setting="default", rmse_holdout=rmse(va0[TARGET], p_b), mae_holdout=mae(va0[TARGET], p_b),
                        rmse_tuning_folds=D6log.loc[D6log.trial == "default", "tune_rmse"].iloc[0]),
                   dict(setting="tuned", rmse_holdout=rmse(va0[TARGET], p_a), mae_holdout=mae(va0[TARGET], p_a),
                        rmse_tuning_folds=best_row.tune_rmse)])
display(D6)
display(D6log.head(8))
add_report("D6 Tuning summary", D6, f"Random search, {N_TRIALS} trials over {list(SPACE)}, time-ordered folds {[str(c.date()) for c in TUNE_CUTS]}. "
           f"Best params: {BEST_PARAMS}")
add_report("D6 Tuning top trials", D6log.head(10))

**✍️ Interpretation:** improvement from tuning (is it large or marginal?), which objective won, whether the best config is at the edge of the search space. …

## D3 · Rolling-origin validation
5 folds, each trains on everything before the cut and validates on the next 14 days — for the **final (tuned) model** and for the **seasonal-naive** baseline. Fold windows are disjoint from the tuning windows.

In [ ]:
final_res, OOF = run_folds(FULL, BEST_PARAMS, "raw", EVAL_CUTS, keep_oof=True)
sn_rows = []
for c in EVAL_CUTS:
    tr, va = split(c)
    sn_rows.append(dict(cut=str(c.date()), rmse_sn=rmse(va[TARGET], seasonal_naive(tr, va, 8)), mae_sn=mae(va[TARGET], seasonal_naive(tr, va, 8))))
D3 = final_res.merge(pd.DataFrame(sn_rows), on="cut").rename(columns={"rmse": "rmse_final", "mae": "mae_final"})
D3["improvement_%"] = 100 * (1 - D3.rmse_final / D3.rmse_sn)
display(D3)
ROLL_MEAN, ROLL_SD = D3.rmse_final.mean(), D3.rmse_final.std(ddof=1)
SN_MEAN, SN_SD = D3.rmse_sn.mean(), D3.rmse_sn.std(ddof=1)
print(f"Final model  RMSE = {ROLL_MEAN:.3f} ± {ROLL_SD:.3f}")
print(f"Seasonal-naive RMSE = {SN_MEAN:.3f} ± {SN_SD:.3f}")
worst = D3.loc[D3.rmse_final.idxmax(), "cut"]
print("worst fold for the final model:", worst)
add_report("D3 Rolling-origin validation", D3,
           f"Final model RMSE {ROLL_MEAN:.2f} ± {ROLL_SD:.2f} vs seasonal-naive {SN_MEAN:.2f} ± {SN_SD:.2f} (mean ± sd over {len(D3)} folds). Worst fold: {worst}.")

**✍️ Interpretation:** what the spread tells you (stable vs. fold-dependent) and *why* the worst fold is worse — e.g. a large unlisted spike, holiday, or outage in that window, or a high-demand period (RMSE scales with demand, so compare `improvement_%` too). …

## D7 · Error analysis
Computed on the out-of-fold predictions of all 5 rolling folds (≈ 20 k zone-hours).

In [ ]:
E = OOF.copy()
E["err"] = E.pred - E[TARGET]
E["abs_err"] = E.err.abs()
E["day_type"] = np.where(E.is_holiday == 1, "holiday", np.where(E.is_weekend == 1, "weekend", "weekday"))
E["has_event"] = (E.ev_n_windows > 0).astype(int)

def breakdown(by):
    g = E.groupby(by)
    out = pd.DataFrame({"n": g.size(), "mean_trips": g[TARGET].mean(),
                        "rmse": g.apply(lambda d: rmse(d[TARGET], d.pred)),
                        "mae": g.abs_err.mean(), "bias(pred-actual)": g.err.mean()})
    out["rmse_%_of_mean"] = 100 * out.rmse / out.mean_trips
    return out.reset_index()

D7_zone, D7_hour, D7_day, D7_event = breakdown("zone"), breakdown("hour"), breakdown("day_type"), breakdown("has_event")
for t, nm in [(D7_zone, "by zone"), (D7_hour, "by hour"), (D7_day, "by day type"), (D7_event, "event window vs none (has_event)")]:
    print("\n==", nm); display(t.round(3))
add_report("D7a Error by zone", D7_zone); add_report("D7a Error by hour", D7_hour)
add_report("D7a Error by day type", D7_day); add_report("D7a Error by event presence", D7_event)

In [ ]:
def active_events(r):
    return ", ".join(c.replace("ev_", "") for c in EVENTS if r[c] == 1) or "none"
top = E.sort_values("abs_err", ascending=False).head(10).copy()
top["active_event_windows"] = top.apply(active_events, axis=1)
top["unexplained_by_calendar"] = np.where(top.ev_n_windows == 0, "YES - no listed event", "no")
D7b = top[["zone", "ts", TARGET, "pred", "err", "day_type", "rain_mm", "active_event_windows", "unexplained_by_calendar"]].copy()
D7b["pred"] = D7b.pred.round(1); D7b["err"] = D7b.err.round(1)
D7b["hypothesis"] = "TODO: write after looking at the row (unlisted event? rain? holiday? outage?)"
display(D7b)
add_report("D7b Ten largest errors", D7b, "Hypotheses to be written by the team after inspection (see notebook).")

**✍️ D7 write-up (fill after running):**
- *Worst zone / hour / day type and why:* …
- *Groups among the top-10 errors* (e.g. all under-predictions in one zone at night → unlisted event; over-predictions on a holiday → holiday response differs by zone): …

## D8 · Response to findings
**Experiment:** does modelling demand *relative to the current level* help (target = `trips / level_28d`, sample-weighted by the level)? This directly targets trend and the Ayat zone (new, lower level). The final model adopts it **only if it wins on the 5 rolling folds**; an honest "it didn't help" is a valid result.

In [ ]:
res_ratio, OOF_ratio = run_folds(FULL, BEST_PARAMS, "ratio", EVAL_CUTS, keep_oof=True)
D8 = pd.DataFrame([dict(variant="raw trips target (current)", rmse_mean=ROLL_MEAN, rmse_sd=ROLL_SD, mae_mean=D3.mae_final.mean()),
                   dict(variant="ratio target (trips / level_28d)", rmse_mean=res_ratio.rmse.mean(), rmse_sd=res_ratio.rmse.std(ddof=1), mae_mean=res_ratio.mae.mean())])
display(D8)
# per-zone effect of the change, to see who benefits
Er = OOF_ratio.assign(err=lambda d: d.pred - d[TARGET])
zr = pd.DataFrame({"rmse_raw": E.groupby("zone").apply(lambda d: rmse(d[TARGET], d.pred)),
                   "rmse_ratio": Er.groupby("zone").apply(lambda d: rmse(d[TARGET], d.pred))})
zr["change_%"] = 100 * (zr.rmse_ratio / zr.rmse_raw - 1)
display(zr.round(3))
FINAL_MODE = "ratio" if D8.rmse_mean[1] < D8.rmse_mean[0] else "raw"
print("\nFINAL TARGET MODE:", FINAL_MODE)
add_report("D8 Response to findings", D8, f"Ratio-target experiment: RMSE {D8.rmse_mean[0]:.3f} -> {D8.rmse_mean[1]:.3f}. Adopted: {FINAL_MODE == 'ratio'}.")
add_report("D8 Per-zone effect of ratio target", zr.reset_index())

**✍️ Interpretation:** what D7 told you, what you changed (add any extra change you try, e.g. a zone-specific tweak), and whether it helped. …

## D9 · Plain-language metric
Holdout = train < 18 Oct, validate 18–31 Oct, final configuration. Also: rolling-origin average.

In [ ]:
m_hold = fit_gbm(tr0, FULL, BEST_PARAMS, FINAL_MODE)
p_hold = predict_gbm(m_hold, va0, FULL, FINAL_MODE)
hold = va0.assign(pred=p_hold)
H_RMSE, H_MAE = rmse(va0[TARGET], p_hold), mae(va0[TARGET], p_hold)
MEAN_DEM = va0[TARGET].mean()
fold_rmse_final = (res_ratio if FINAL_MODE == "ratio" else final_res)
R_MEAN, R_SD = fold_rmse_final.rmse.mean(), fold_rmse_final.rmse.std(ddof=1)
TPD = 1.3                                              # trips per active driver-hour (from the history)
para = (f"On the 18-31 Oct holdout the model is typically off by {H_MAE:.1f} trips per zone-hour (MAE) and its RMSE is {H_RMSE:.1f} trips, "
        f"i.e. {100*H_MAE/MEAN_DEM:.0f}% and {100*H_RMSE/MEAN_DEM:.0f}% of the average demand of {MEAN_DEM:.1f} trips per zone-hour. "
        f"At about {TPD} trips per driver-hour, that is roughly {H_MAE/TPD:.1f} drivers (MAE) or {H_RMSE/TPD:.1f} drivers (RMSE) per zone-hour. "
        f"Across {len(fold_rmse_final)} rolling folds RMSE is {R_MEAN:.1f} ± {R_SD:.1f}. "
        f"Practically: plan driver supply per zone-hour with a buffer of about {np.ceil(H_RMSE/TPD):.0f} drivers, and expect larger misses at event hours.")
print(para)
D9 = pd.DataFrame([dict(metric="RMSE", trips_per_zone_hour=H_RMSE, pct_of_mean_demand=100*H_RMSE/MEAN_DEM, approx_drivers=H_RMSE/TPD),
                   dict(metric="MAE",  trips_per_zone_hour=H_MAE,  pct_of_mean_demand=100*H_MAE/MEAN_DEM,  approx_drivers=H_MAE/TPD)])
display(D9)
add_report("D9 Plain-language metric", D9, para)

### Extra · Weather-forecast stress test
Validation uses *observed* weather, but November rows are *forecasts* (noisier). We perturb the validation weather (temperature ±1.5 °C, rain × lognormal noise) to see how much accuracy could be lost in production.

In [ ]:
rng = np.random.default_rng(SEED)
v = va0.copy()
tsu = pd.Index(v.ts.unique())
mult = pd.Series(np.exp(rng.normal(0, 0.5, len(tsu))), index=tsu)
dT   = pd.Series(rng.normal(0, 1.5, len(tsu)), index=tsu)
for c in ["rain_mm", "rain_3h", "rain_6h", "rain_24h"]:
    v[c] = v[c] * v.ts.map(mult)
v["temp_c"] = v.temp_c + v.ts.map(dT)
v["rain_class"] = np.digitize(v.rain_mm, [0.1, 2.5, 7.6])
stress = pd.DataFrame([dict(weather="observed (clean)", rmse=H_RMSE, mae=H_MAE),
                       dict(weather="noisy forecast-like", rmse=rmse(v[TARGET], predict_gbm(m_hold, v, FULL, FINAL_MODE)),
                            mae=mae(v[TARGET], predict_gbm(m_hold, v, FULL, FINAL_MODE)))])
display(stress)
add_report("Extra weather forecast stress test", stress, "Noise levels are assumptions, not measured forecast error; treat as a sensitivity check.")

## Final model: fit on all history → predict 1–14 Nov → save
Fitted on the **entire** train table (1 Jan – 31 Oct). Nothing is fitted on the test file; the test rows come from the same pipeline (Deliverable A) with statistics learned from train.

In [ ]:
final_model = fit_gbm(train, FULL, BEST_PARAMS, FINAL_MODE)
pred_test = predict_gbm(final_model, test, FULL, FINAL_MODE)

bundle = {"model": final_model, "features": FULL, "zones": ZONES, "target_mode": FINAL_MODE,
          "params": BEST_PARAMS, "train_range": (str(train.ts.min()), str(train.ts.max())),
          "holdout_rmse_18oct": H_RMSE, "rolling_rmse_mean": R_MEAN, "rolling_rmse_sd": R_SD,
          "lightgbm_version": lgb.__version__, "note": "predict: X=df[['zone']+features] with zone as Categorical(categories=zones); "
                                                       "if target_mode=='ratio' multiply by df.level_28d.clip(lower=1); clip at 0."}
joblib.dump(bundle, MODELS / "final_model.joblib", compress=3)
print("saved", MODELS / "final_model.joblib", f"({(MODELS / 'final_model.joblib').stat().st_size/1e6:.1f} MB)")

# reload and verify identical predictions (the demo will do exactly this)
b2 = joblib.load(MODELS / "final_model.joblib")
p2 = predict_gbm(b2["model"], test, b2["features"], b2["target_mode"])
assert np.allclose(pred_test, p2), "reloaded model gives different predictions"
print("reload check: PASS")

In [ ]:
# ---------- submission file (G4)
sub = pd.DataFrame({"row_id": test["row_id"].values, "predicted_trips": pred_test})
tmpl_path = ROOT / "data/raw/submission_template.csv"
if tmpl_path.exists():                                   # keep the original order of the template
    tmpl = pd.read_csv(tmpl_path)
    assert set(tmpl.row_id) == set(sub.row_id), "row_ids differ from the template"
    sub = tmpl[["row_id"]].merge(sub, on="row_id", how="left")
out_path = SUB / f"team_{TEAM_NAME}_submission.csv"
sub.to_csv(out_path, index=False)

chk = pd.read_csv(out_path)
checks = {"columns are row_id, predicted_trips": list(chk.columns) == ["row_id", "predicted_trips"],
          "4,032 rows": len(chk) == 4032,
          "row_ids unique": chk.row_id.is_unique,
          "no blanks / NaN": chk.predicted_trips.notna().all(),
          "numeric": pd.api.types.is_numeric_dtype(chk.predicted_trips),
          "no negatives": (chk.predicted_trips >= 0).all()}
for k, ok in checks.items():
    print(("PASS" if ok else "FAIL"), "-", k)
assert all(checks.values())
print("\nprediction summary:", chk.predicted_trips.describe().round(2).to_dict())
print("train mean (Oct):", round(train[train.ts >= "2025-10-01"][TARGET].mean(), 2), "| predicted Nov mean:", round(chk.predicted_trips.mean(), 2))

**Sanity check:** the predicted November mean should be close to October's level (≈ 33 trips/zone-hour) — if it is wildly different, something is wrong.

## Figures for Deliverable C owned by modeling (fig10, fig11, fig12)

In [ ]:
# ---------- fig10: model comparison
bars = D2[["model", "rmse"]].copy()
bars = pd.concat([bars, pd.DataFrame([dict(model="LightGBM tuned (holdout)", rmse=H_RMSE),
                                      dict(model="LightGBM tuned (5-fold mean)", rmse=R_MEAN)])], ignore_index=True)
fig, ax = plt.subplots(figsize=(11, 5.5))
colors = [OKABE[0] if "tuned" in m else OKABE[5] for m in bars.model]
ax.bar(range(len(bars)), bars.rmse, color=colors, edgecolor="black", linewidth=0.5)
ax.errorbar([len(bars) - 1], [R_MEAN], yerr=[R_SD], fmt="none", ecolor="black", capsize=6, lw=2, label="±1 sd across rolling folds")
ax.axhline(BASE_MEAN_RMSE, color=OKABE[4], ls="--", lw=2, label=f"Baseline: mean predictor ({BASE_MEAN_RMSE:.1f})")
ax.axhline(BASE_SN_RMSE, color=OKABE[1], ls="--", lw=2, label=f"Baseline: seasonal-naive, last 8 wk ({BASE_SN_RMSE:.1f})")
for i, v_ in enumerate(bars.rmse):
    ax.text(i, v_ + 0.3, f"{v_:.1f}", ha="center", fontsize=9)
ax.set_xticks(range(len(bars))); ax.set_xticklabels(bars.model, rotation=25, ha="right")
ax.set_ylim(0, max(bars.rmse.max(), BASE_MEAN_RMSE) * 1.15)
ax.set_ylabel("Validation RMSE (trips per zone-hour)")
ax.set_title("Model comparison on 18-31 Oct 2025 holdout (lower is better)")
ax.legend(loc="upper right"); fig.tight_layout()
fig.savefig(FIGS / "fig10_model_comparison.png"); plt.show()

In [ ]:
# ---------- fig11: forecast vs actual, 3 zones, one full validation fortnight (18-31 Oct)
by_mean = hold.groupby("zone")[TARGET].mean().sort_values(ascending=False)
zones3 = [by_mean.index[0], by_mean.index[len(by_mean) // 2], "Ayat" if "Ayat" in by_mean.index else by_mean.index[-1]]
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
for ax, z in zip(axes, zones3):
    d = hold[hold.zone == z].sort_values("ts")
    ax.plot(d.ts, d[TARGET], color=OKABE[7], lw=1.1, label="Actual")
    ax.plot(d.ts, d.pred, color=OKABE[4], lw=1.1, label="Predicted")
    ev = d[d.ev_n_windows > 0]
    ax.scatter(ev.ts, ev[TARGET], s=18, color=OKABE[2], zorder=3, label="Event window active")
    ax.set_title(f"{z}  -  RMSE {rmse(d[TARGET], d.pred):.1f}, MAE {mae(d[TARGET], d.pred):.1f} trips/hour")
    ax.set_ylabel("Trips per hour"); ax.legend(loc="upper left", ncol=3)
axes[-1].set_xlabel("Date / hour (Addis Ababa time, EAT)")
fig.suptitle("Forecast vs actual hourly trips, 18-31 Oct 2025 (model trained only on data before 18 Oct)", y=0.995, fontsize=13)
fig.tight_layout(); fig.savefig(FIGS / "fig11_forecast_vs_actual.png"); plt.show()

In [ ]:
# ---------- fig12: permutation importance (RMSE increase when a feature is shuffled) on the holdout
def perm_importance(m, df, feats, mode, n_repeats=5):
    rng = np.random.default_rng(SEED)
    base = rmse(df[TARGET], predict_gbm(m, df, feats, mode))
    out = []
    for f in ["zone"] + list(feats):
        d_ = []
        for _ in range(n_repeats):
            x = df.copy(); x[f] = rng.permutation(x[f].values)
            d_.append(rmse(x[TARGET], predict_gbm(m, x, feats, mode)) - base)
        out.append(dict(feature=f, importance=np.mean(d_), sd=np.std(d_)))
    return pd.DataFrame(out).sort_values("importance", ascending=False).reset_index(drop=True)

def group_of(f):
    if f in WEATHER: return "Weather"
    if f in EVENTS or f in ("is_holiday", "is_holiday_eve", "is_school_break"): return "Events / calendar-from-events"
    if f in TREND: return "Trend / lag"
    if f == "zone": return "Zone"
    return "Calendar"

PI = perm_importance(m_hold, va0, FULL, FINAL_MODE, n_repeats=2 if FAST else 5)
PI["group"] = PI.feature.map(group_of)
add_report("D-fig12 Permutation importance", PI.round(4))
gcol = {"Weather": OKABE[5], "Events / calendar-from-events": OKABE[4], "Trend / lag": OKABE[2], "Zone": OKABE[3], "Calendar": "#999999"}
topn = PI.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(topn.feature, topn.importance, xerr=topn.sd, color=[gcol[g] for g in topn.group], edgecolor="black", linewidth=0.4, capsize=3)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=g) for g, c in gcol.items() if g in set(topn.group)], loc="lower right", title="Feature group")
ax.set_xlabel("Increase in holdout RMSE when feature is shuffled (trips per zone-hour)")
ax.set_title("Permutation importance, top 15 features (weather and event features coloured)")
fig.tight_layout(); fig.savefig(FIGS / "fig12_feature_importance.png"); plt.show()
display(PI.head(15).round(3))
print("\nbest weather feature:", PI[PI.group == "Weather"].iloc[0].to_dict())
print("best event feature  :", PI[PI.group.str.startswith("Events")].iloc[0].to_dict())

In [ ]:
# ---------- captions for fig10-12 (merge these three entries into figures/figure_captions.md)
bw = PI[PI.group == "Weather"].iloc[0]; be = PI[PI.group.str.startswith("Events")].iloc[0]
cap = f'''
**fig10_model_comparison.png** - Validation RMSE of every model on the 18-31 Oct 2025 holdout, with both baselines as dashed lines and rolling-origin error bars on the final model. The tuned LightGBM reaches RMSE {H_RMSE:.1f} (rolling mean {R_MEAN:.1f} ± {R_SD:.1f}) versus {BASE_SN_RMSE:.1f} for seasonal-naive.

**fig11_forecast_vs_actual.png** - Predicted vs actual hourly trips for {', '.join(zones3)} over 18-31 Oct 2025 (green dots = hours inside a listed event window). The model follows the daily and weekly rhythm closely; remaining misses concentrate at spikes.

**fig12_feature_importance.png** - Permutation importance of the 15 most important features on the holdout; weather and event features are coloured. Top feature: {PI.feature[0]}; strongest weather feature: {bw.feature}; strongest event feature: {be.feature}.
'''
(FIGS / "captions_fig10_12.md").write_text(cap.strip() + "\n", encoding="utf-8")
print(cap)

## Write `reports/D_model_evaluation.md`

In [ ]:
lines = [f"# Deliverable D - Modeling & Evaluation (team {TEAM_NAME})", "",
         f"*Final model:* LightGBM (target mode: `{FINAL_MODE}`), fitted on 1 Jan - 31 Oct 2025. "
         f"*Holdout split:* train < {MAIN_CUT.date()}, validate {va0.ts.min().date()} - {va0.ts.max().date()}. "
         f"*Rolling-origin cuts:* {', '.join(str(c.date()) for c in EVAL_CUTS)}. *Tuning cuts:* {', '.join(str(c.date()) for c in TUNE_CUTS)}. "
         f"*Seed:* {SEED}. No score is computed on the test file.", ""]
for title, df, text in REPORT:
    lines += [f"## {title}", ""]
    if text: lines += [text, ""]
    if df is not None: lines += [md_table(df.head(40)), ""]
lines += ["## Team interpretation (fill in)", "", "_Copy your interpretations from the notebook's ✍️ cells here._", ""]
(REPORTS / "D_model_evaluation.md").write_text("\n".join(lines), encoding="utf-8")
print("wrote", REPORTS / "D_model_evaluation.md")
print("\nHEADLINE: holdout RMSE %.2f | rolling RMSE %.2f ± %.2f | seasonal-naive %.2f | winner family: %s" % (H_RMSE, R_MEAN, R_SD, BASE_SN_RMSE, WINNER))

---
### Checklist after running
- [ ] Replace `TEAM_NAME` at the top and re-run so the submission file name is right.
- [ ] Fill every **✍️ Interpretation** cell (a number without interpretation earns half marks) and copy them into `reports/D_model_evaluation.md`.
- [ ] Write the hypotheses in the D7 top-10 table.
- [ ] Merge `figures/captions_fig10_12.md` into `figures/figure_captions.md`.
- [ ] Pin versions printed in the setup cell into `requirements.txt` (`lightgbm`, `scikit-learn`, `pandas`, `numpy`, `matplotlib`, `joblib`).
- [ ] Demo app: load `models/final_model.joblib` → `bundle["model"]`, `bundle["features"]`, `bundle["zones"]`, `bundle["target_mode"]` (see the `note` key).